<a href="https://colab.research.google.com/github/t4ylorr37/SBI-NPE_IMF_Inference/blob/main/notebooks/SBI_NPE_IMF_Inference_env_config.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount google drive to google colab

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


drive is accessible through /content/drive/MyDrive - mount again whenever new runtime requires authorization

Establishing drive paths for data generated in colab to be stored in:

In [2]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/SBI-NPE_IMF_Inference"
)
COLAB_ROOT = Path("/content")

SIMULATION_DATA = DRIVE_ROOT / "Simulation data"
POSTERIORS = DRIVE_ROOT / "Posteriors"
NOTEBOOKS = DRIVE_ROOT / "Notebooks"
MODELS = DRIVE_ROOT / "Models"
MODEL_CHECKPOINTS = DRIVE_ROOT / "Model checkpoints"
LOGS = DRIVE_ROOT / "Logs"
FIGURES = DRIVE_ROOT / "Figures"
EMPIRICAL_DATA = DRIVE_ROOT / "Empirical data"
DIAGNOSTICS = DRIVE_ROOT / "Diagnostics"
#Establish a temporary root for data to be stored in, so it can be periodically transferred to google drive in batched rather than all at once
TEMP_ROOT = COLAB_ROOT / "Temp_dir"

for folder in [
    SIMULATION_DATA,
    POSTERIORS,
    NOTEBOOKS,
    MODELS,
    MODEL_CHECKPOINTS,
    LOGS,
    FIGURES,
    EMPIRICAL_DATA,
    DIAGNOSTICS,
    TEMP_ROOT
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories established")

Project directories established


## Connect github to google colab

In [3]:
!git clone https://github.com/t4ylorr37/SBI-NPE_IMF_Inference.git

Cloning into 'SBI-NPE_IMF_Inference'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 117 (delta 54), reused 12 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 40.22 KiB | 3.09 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [4]:
%cd /content/SBI-NPE_IMF_Inference

/content/SBI-NPE_IMF_Inference


colab is working in the github repository, its contents can be inspected with:

!1s

When code is changed in VSCode and pushed to github:

!git pull origin main

in colab

## Install python requirements and configure python-FSPS

In [6]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 545.2/545.2 kB 5.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.3/950.3 kB 37.5 MB/s  0:00:00
  Created wheel for nflows: filename=nflows-0.14-py3-none-any.whl size=53703 sha256=1c546eca997e431ca3b2cdb83eb630347a380b5273dfd8f3654ec7e273387756
  Stored in directory: /root/.cache/pip/wheels/74/99/00/66d5b7f05da2e381ac0eacaaa0c373494d0009e2af5db16cd9
Successfully built nflows
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [jupyter]


Install python-FSPS, fsps source files in content/fsps

In [7]:
!apt-get update -qq
!apt-get install -y gfortran

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gfortran is already the newest version (4:13.2.0-7ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.


In [8]:
!git clone https://github.com/cconroy20/fsps.git /content/fsps

Cloning into '/content/fsps'...
remote: Enumerating objects: 3583, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 3583 (delta 2), reused 2 (delta 2), pack-reused 3576 (from 2)
Receiving objects: 100% (3583/3583), 2.18 GiB | 19.50 MiB/s, done.
Resolving deltas: 100% (2157/2157), done.
Updating files: 100% (384/384), done.


In [9]:
import os
os.environ["SPS_HOME"] = "/content/fsps"

In [10]:
!FFLAGS="-DAFE_FLAG=1" pip install --no-binary=fsps fsps

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fsps: filename=fsps-0.5.0-cp313-cp313-linux_x86_64.whl size=248209 sha256=50ec1a471e0d43c170557989f8374579199aa703f16827cc6fb396795aba9a2e
  Stored in directory: /root/.cache/pip/wheels/6a/ac/48/681d2854a24a60cd54d3e51ae35b206db1b4b7a2fcc5e14a14
Successfully built fsps


Check that packages work and fsps generates spectra

In [11]:
import numpy as np
import scipy
import matplotlib
import h5py
import astropy
import torch
import sbi
import fsps

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("pytorch:", torch.__version__)
print("sbi:", sbi.__version__)
print("h5py:", h5py.__version__)
print("astropy:", astropy.__version__)
print("python-fsps", fsps.__version__)

sp = fsps.StellarPopulation(zcontinuous=1, imf_type=2, tage=5.0)
sp.params["afe"] = 0.0
wavelength, luminosity = sp.get_spectrum()

print("stellar libraries:", sp.libraries)
print("wavelength shape:", wavelength.shape)
print("spectrum shape:", luminosity.shape)

#make sure alpha enhancement is working
sp.params["afe"] = 0.3
wavelength, alpha_luminosity = sp.get_spectrum()

difference = np.max(np.abs(alpha_luminosity - luminosity))
print("max flux difference with alpha enhancement:", difference)

numpy: 2.4.6
scipy: 1.16.3
pytorch: 2.14.1+cpu
sbi: 0.27.0
h5py: 3.14.0
astropy: 7.2.3
python-fsps 0.5.0
stellar libraries: (b'mist', b'c3k_lr', b'DL07')
wavelength shape: (1936,)
spectrum shape: (107, 1936)
max flux difference with alpha enhancement: 1.1092102272264968e-13


once everything works, save the environment information

In [12]:
!pip freeze > /content/requirements-frozen.txt

copy into drive to save specific packages the model uses

In [13]:
import shutil

shutil.copy(
    "/content/requirements-frozen.txt",
    str(DRIVE_ROOT / "requirements-frozen.txt")
)

'/content/drive/MyDrive/SBI-NPE_IMF_Inference/requirements-frozen.txt'



---

In [ ]:
!git config --global user.email "taylorcurboy@icloud.com"
!git config --global user.name "t4ylorr37"

## Conflig cell to run at the beginning of every notebook

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/SBI-NPE_IMF_Inference"
)
COLAB_ROOT = Path("/content")

SIMULATION_DATA = DRIVE_ROOT / "Simulation data"
POSTERIORS = DRIVE_ROOT / "Posteriors"
NOTEBOOKS = DRIVE_ROOT / "Notebooks"
MODELS = DRIVE_ROOT / "Models"
MODEL_CHECKPOINTS = DRIVE_ROOT / "Model checkpoints"
LOGS = DRIVE_ROOT / "Logs"
FIGURES = DRIVE_ROOT / "Figures"
EMPIRICAL_DATA = DRIVE_ROOT / "Empirical data"
DIAGNOSTICS = DRIVE_ROOT / "Diagnostics"
#Establish a temporary root for data to be stored in, so it can be periodically transferred to google drive in batched rather than all at once
TEMP_ROOT = COLAB_ROOT / "Temp_dir"

for folder in [
    SIMULATION_DATA,
    POSTERIORS,
    NOTEBOOKS,
    MODELS,
    MODEL_CHECKPOINTS,
    LOGS,
    FIGURES,
    EMPIRICAL_DATA,
    DIAGNOSTICS,
    TEMP_ROOT
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories established")

!git clone https://github.com/t4ylorr37/SBI-NPE_IMF_Inference.git
%cd /content/SBI-NPE_IMF_Inference

!pip install -r requirements-frozen.txt

To install fsps:

In [ ]:
import os

!apt-get update -qq
!apt-get install -y gfortran

!git clone https://github.com/cconroy20/fsps.git /content/fsps

os.environ["SPS_HOME"] = "/content/fsps"
!FFLAGS="-DAFE_FLAG=1" pip install --no-binary=fsps fsps